In [1]:
import os

PROJECT = "/content/warp_reduction"
SRC = os.path.join(PROJECT, "src")
os.makedirs(SRC, exist_ok=True)

In [2]:
%%writefile /content/warp_reduction/src/warp_shuffle.cu
#include <iostream>
#include <vector>
#include <cuda_runtime.h>

#define CHECK(call) do { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        std::cerr << "CUDA error: " \
                  << cudaGetErrorString(err) \
                  << std::endl; \
        return 1; \
    } \
} while (0)

__global__ void reduceWarp(
    const int* input,
    int* output
) {
    int lane = threadIdx.x;
    int value = input[lane];

    // All 32 lanes participate in every shuffle.
    unsigned int mask = 0xffffffff;

    for (int offset = 16; offset > 0; offset /= 2) {
        int other = __shfl_down_sync(
            mask, value, offset
        );
        value += other;
    }

    // Lane 0 contains the total.
    if (lane == 0) {
        output[0] = value;
    }
}

int main() {
    const int n = 32;

    std::vector<int> h_input(n);
    for (int i = 0; i < n; i++) {
        h_input[i] = i + 1;
    }

    int h_output = 0;
    int *d_input = nullptr;
    int *d_output = nullptr;

    CHECK(cudaMalloc(&d_input, n * sizeof(int)));
    CHECK(cudaMalloc(&d_output, sizeof(int)));

    CHECK(cudaMemcpy(
        d_input, h_input.data(), n * sizeof(int),
        cudaMemcpyHostToDevice
    ));

    // Exactly one warp: 32 threads.
    reduceWarp<<<1, 32>>>(d_input, d_output);

    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    CHECK(cudaMemcpy(
        &h_output, d_output, sizeof(int),
        cudaMemcpyDeviceToHost
    ));

    std::cout << "Warp-shuffle sum: "
              << h_output << std::endl;

    CHECK(cudaFree(d_input));
    CHECK(cudaFree(d_output));

    return 0;
}

Writing /content/warp_reduction/src/warp_shuffle.cu


In [3]:
!nvcc /content/warp_reduction/src/warp_shuffle.cu \
    -o /content/warp_reduction/src/warp_shuffle

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [4]:
!/content/warp_reduction/src/warp_shuffle

Warp-shuffle sum: 528
